# MLLM Inference & Evaluation Pipeline for Abstract Visual Reasoning

This notebook serves as the core inference and evaluation engine for benchmarking Multimodal Large Language Models (MLLMs) on the abstract visual reasoning dataset. 

### 🎯 Core Objectives
- **Multi-Model Batch Inference:** Automates the process of encoding local images to base64 and sequentially querying various state-of-the-art MLLM APIs.
- **Dataset Categorization:** Maps specific image ID ranges to their corresponding knowledge domains (e.g., *Public Figures, Popular Media, Literary Works*) and specific reasoning tasks (e.g., *Which Movie?, Guess the Idiom*).
- **Visual Style Classification:** Uses few-shot LLM prompting to automatically categorize the artistic rendering style of the dataset images (e.g., *Illustration, Cartoon, Caricature, Emoji Combination, Minimalism*).
- **Performance Aggregation:** Analyzes the compiled JSON results to calculate cross-model success rates (e.g., calculating the frequency of images recognized by 0 to 6 models to identify the baseline "capability floor").

### 🤖 Supported Models Evaluated
The notebook contains configured API clients and batching logic for:
* **OpenAI:** GPT-4o
* **Anthropic:** Claude 3.5 Sonnet
* **Google:** Gemini 2.0 Flash
* **ZhipuAI:** GLM-4v-plus
* **DeepInfra (Open-Source):** Llama 3.2 90B Vision, Gemma-3 27B
* **DeepSeek:** DeepSeek-Chat

### 📁 Input / Output
* **Inputs:** Local image directories (`.jpg`, `.png`) and structured JSON files (`result_3.json`).
* **Outputs:** Inference JSON files (`infer.json`, `view.json`, `gemma_view.json`) containing model predictions, aggregated accuracy statistics, and style classifications.


      

In [84]:
import base64

def image_to_base64(image_path):
    try:
        with open(image_path, "rb") as image_file:
            # Read the image binary data
            binary_data = image_file.read()
            # Encode binary data to base64
            base64_data = base64.b64encode(binary_data)
            # Convert bytes to string
            base64_string = base64_data.decode('utf-8')
            # Create the complete base64 URL
            base64_url = f"data:image/{image_path.split('.')[-1]};base64,{base64_string}"
            return base64_url
    except Exception as e:
        return f"Error: {str(e)}"

In [88]:
from openai import OpenAI
import httpx

client = OpenAI(
    base_url="https://api.xty.app/v1", 
    api_key=key,
    http_client=httpx.Client(
        base_url="https://api.xty.app/v1",
        follow_redirects=True,
    ),
)
image_path = "3429.jpg" 
completion = client.chat.completions.create(
  model="claude-3-5-sonnet-20241022",
  messages=[
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "Who is this?"
          },
          {
            "type": "image_url",
            "image_url": {"url":image_to_base64(image_path)}
          }
        ]
      }
    ]
)

print(completion.choices[0].message.content)

This appears to be Pandora from Greek mythology, depicted in the classic black-figure style of ancient Greek pottery art. According to the myth, she was given a box (though it was actually a jar or "pithos" in the original Greek) by the gods, which she opened, releasing all the world's evils. The image shows a female figure in a long flowing dress holding up what appears to be the infamous box, with wispy shapes emerging from it, represented in a traditional Ancient Greek artistic style against an orange background with classical columns.


In [1]:
# Assume openai>=1.0.0
from openai import OpenAI

# Create an OpenAI client with your deepinfra token and endpoint
openai = OpenAI(
    api_key="xxxx",
    base_url="https://api.deepinfra.com/v1/openai",
)

chat_completion = openai.chat.completions.create(
    model="meta-llama/Llama-3.2-90B-Vision-Instruct",
    messages=[{"role": "user", "content": "Hello"}],
)

print(chat_completion.choices[0].message.content)
print(chat_completion.usage.prompt_tokens, chat_completion.usage.completion_tokens)

# Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?
# 11 25


Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?
11 24


## GPT

## Claude

In [81]:
from openai import OpenAI
import httpx

client = OpenAI(
    base_url="https://api.xty.app/v1", 
    api_key="xxx",
    http_client=httpx.Client(
        base_url="https://api.xty.app/v1",
        follow_redirects=True,
    ),
)

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "can you describe a given image?"}
  ]
)

print(completion.choices[0].message.content)

I'm unable to view images directly. However, if you describe an image to me, I can help with analysis, provide information, or answer questions about it! Let me know how I can assist you.


In [109]:
import os
import base64
import json
from openai import OpenAI
import requests

def process_images(folder_path):
    client = OpenAI(
    base_url="https://api.xty.app/v1", 
    api_key="xxx",
    http_client=httpx.Client(
        base_url="https://api.xty.app/v1",
        follow_redirects=True,
    ),
    )
    output_file = os.path.join(folder_path, 'infer.json')

    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
    else:
        results = {}
    
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    for img_file in image_files:
        key = os.path.splitext(img_file)[0]
        
        # Skip if already processed
        if key in results:
            print(f"Skipping {img_file} - already processed")
            continue
            
        img_path = os.path.join(folder_path, img_file)
        
        try:
            # Read and encode image with proper data URI format
            with open(img_path, 'rb') as f:
                img_base = base64.b64encode(f.read()).decode('utf-8')
                # Add the appropriate data URI prefix based on file extension
                if img_file.lower().endswith(('.jpg', '.jpeg')):
                    img_base = f"data:image/jpeg;base64,{img_base}"
                elif img_file.lower().endswith('.png'):
                    img_base = f"data:image/png;base64,{img_base}"
                    

            completion = client.chat.completions.create(
              model="gemini-2.0-flash-exp",
              messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content":[{"type": "image_url", "image_url": {"url": img_base}},
                                            {"type": "text", "text": "Concisely answer the question on the image!"}]}
              ]
            )
            value = completion.choices[0].message.content
            results[key] = value
            print(key,'|',value)

            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            results[key] = "Error: " + str(e)
            # Save results even if there's an error
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
    
    return output_file

# Usage
folder_path = "a"
output_file = process_images(folder_path)
print(f"Results saved to: {output_file}")

0650 | Tom Hanks
Results saved to: a\infer.json


## Gemini

In [33]:
import os
import base64
import json
from openai import OpenAI
import requests

def process_images(folder_path):
    url = "https://api.ainewserver.com/v1/chat/completions"
    headers = {
        "Authorization": "xxx",
        "content-type": "application/json"
    }
    output_file = os.path.join(folder_path, 'test.json')
    
    # Load existing results if file exists
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
    else:
        results = {}
    
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    for img_file in image_files:
        key = os.path.splitext(img_file)[0]
        
        # Skip if already processed
        if key in results:
            print(f"Skipping {img_file} - already processed")
            continue
            
        img_path = os.path.join(folder_path, img_file)
        
        try:
            # Read and encode image with proper data URI format
            with open(img_path, 'rb') as f:
                img_base = base64.b64encode(f.read()).decode('utf-8')
                # Add the appropriate data URI prefix based on file extension
                if img_file.lower().endswith(('.jpg', '.jpeg')):
                    img_base = f"data:image/jpeg;base64,{img_base}"
                elif img_file.lower().endswith('.png'):
                    img_base = f"data:image/png;base64,{img_base}"
                    
            data = {
                "messages":[
                    {
                        "role": "user",
                        "content":[{"type": "image_url", "image_url": {"url": img_base}},
                                   {"type": "text", "text": "concisely answer the question on the image"}]
                    }],
                "model": "claude-3-5-sonnet-20241022"
            }
         
            response = requests.post(url, headers=headers, json=data)
            data = json.loads(response.text)
            value = data['choices'][0]['message']['content']   
            results[key] = value
            print(key,'|',value)
            
            # Save results after each successful processing
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            results[key] = "Error: " + str(e)
            # Save results even if there's an error
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
    
    return output_file

# Usage
folder_path = "test"
output_file = process_images(folder_path)
print(f"Results saved to: {output_file}")

0001 | I don't see a question in the image - it appears to be a large grid of cells or boxes. Could you please let me know what specific question you'd like me to answer about this image?
Error processing 2324.jpg: 'choices'
Error processing 2744.jpg: 'choices'
3058 | I don't see a specific question in the image. The image appears to be a large block of text or code in a dark color against a light background. Could you please clarify what question you would like me to answer about this image?
3429 | I apologize, but I'm unable to see any question in the image. The image appears to be a large table or grid with multiple cells, but I cannot make out any specific question that needs to be answered. Could you please share the question you'd like me to help with?
Results saved to: test\test.json


## Deepinfra(Llama & Qwen)

In [16]:
import os
import base64
import json
from openai import OpenAI

def process_images(folder_path):
    openai = OpenAI(
    api_key="xxxxxxxxxxx",
    base_url="https://api.deepinfra.com/v1/openai",
    )
    results = {}  
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    for img_file in image_files:
        key = os.path.splitext(img_file)[0]
        img_path = os.path.join(folder_path, img_file)
        
        try:
            # Read and encode image with proper data URI format
            with open(img_path, 'rb') as f:
                img_base = base64.b64encode(f.read()).decode('utf-8')
                # Add the appropriate data URI prefix based on file extension
                if img_file.lower().endswith(('.jpg', '.jpeg')):
                    img_base = f"data:image/jpeg;base64,{img_base}"
                elif img_file.lower().endswith('.png'):
                    img_base = f"data:image/png;base64,{img_base}"
            
            response = openai.chat.completions.create(
                model="google/gemma-3-27b-it",
                messages=[{
                    "role": "user", 
                    "content":[
                        {"type": "image_url", "image_url": {"url": img_base}},
                        {"type": "text", "text": "concisely answer the question on the image"}
                    ]
                }],
            )
                        
            value = response.choices[0].message.content
            results[key] = value
            print(key,'|',value)
            
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            results[key] = "Error: " + str(e)
    
    output_file = os.path.join(folder_path, 'gemma_view.json')
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    return output_file

# Usage
folder_path = "img"
output_file = process_images(folder_path)
print(f"Results saved to: {output_file}")

Error processing 0001.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0002.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0003.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0004.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0005.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0006.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0007.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0008.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error processing 0009.jpg: Error code: 405 - {'detail': 'Multimodal is not supported for model: {inp.model}'}
Error proc

KeyboardInterrupt: 

In [ ]:
import os
import base64
import json
from openai import OpenAI

def process_images(folder_path):
    openai = OpenAI(
    api_key="xxxxxxx",
    base_url="https://api.deepinfra.com/v1/openai",
    )
    
    output_file = os.path.join(folder_path, 'view.json')
    
    # Initialize results from existing JSON file if it exists
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
    else:
        results = {}
    
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    for img_file in image_files:
        key = os.path.splitext(img_file)[0]
        img_path = os.path.join(folder_path, img_file)
        
        try:
            # Read and encode image with proper data URI format
            with open(img_path, 'rb') as f:
                img_base = base64.b64encode(f.read()).decode('utf-8')
                # Add the appropriate data URI prefix based on file extension
                if img_file.lower().endswith(('.jpg', '.jpeg')):
                    img_base = f"data:image/jpeg;base64,{img_base}"
                elif img_file.lower().endswith('.png'):
                    img_base = f"data:image/png;base64,{img_base}"
            
            response = openai.chat.completions.create(
                model="google/gemma-3-27b-it",
                messages=[{
                    "role": "user", 
                    "content":[
                        {"type": "image_url", "image_url": {"url": img_base}},
                        {"type": "text", "text": "concisely answer the question on the image"}
                    ]
                }],
            )
                        
            value = response.choices[0].message.content
            results[key] = value
            print(key,'|',value)
            
            # Save to JSON file after each new pair is added
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            results[key] = "Error: " + str(e)
            
            # Still save to JSON file if there's an error
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
    
    return output_file

# Usage
folder_path = "img"
output_file = process_images(folder_path)
print(f"Results saved to: {output_file}")

## GLM

## Other

In [ ]:
import json
data = json.loads(response.text)
print(data['choices'][0]['message']['content'])

In [6]:
# Please install OpenAI SDK first: `pip3 install openai`

from openai import OpenAI

client = OpenAI(api_key="xxxx", base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello,can you process image"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Hello! Unfortunately, I cannot process or analyze images directly. However, if you describe the image or provide details about it, I can certainly help interpret or provide information based on your description. Let me know how I can assist!


In [119]:
import json

# Function to sum binary lists
def sum_binary_lists(data_dict):
    # Initialize result list with zeros
    result = [0] * 6  # Assuming all binary lists have length 6
    
    # Iterate through all items in dictionary
    for item in data_dict.values():
        binary_list = item[1]  # Get the binary list (second element)
        # Sum corresponding elements
        for i in range(len(binary_list)):
            result[i] += binary_list[i]
    
    return result

# Read from JSON file
with open('json/result_3.json', 'r') as f:
    data = json.load(f)

# Calculate sum
result = sum_binary_lists(data)
print(result)
# Write result to JSON file
# with open('output.json', 'w') as f:
#     json.dump(result, f)

[2418, 1929, 2749, 1807, 1777, 1066]


In [121]:
import json

def count_third_values(data_dict):
    # Initialize result list with zeros for numbers 0-6 (7 positions)
    result = [0] * 7
    
    # Count frequency of each number
    for item in data_dict.values():
        third_value = item[2]  # Get the third value
        result[third_value] += 1
    
    return result

# Read from JSON file
with open('json/result_3.json', 'r') as f:
    data = json.load(f)

# Calculate frequency
result = count_third_values(data)
print(result)

[285, 436, 516, 555, 586, 637, 514]


number of  0 to 6 recognitions:[285, 436, 516, 555, 586, 637, 514]. from this result ,we can further look into following interetsting points:
- for the number of 0 recognition(285),which means the number of images that all the LLMs fails to recognise, so when evaluating the recognition performance, it's better to evlauate against the number of at least one recognition(3529-285).
- for the total number of 1 recogintion(436),we can further look into ,each LLM contribute how much of these number.
- for the total number of 5 recognition(637),another interesting point is ,each LLM contribute how much to this number, which means, each LLM occupy how much percentage of in being the only one LLM that failed the specific task

## print out the presentation method

In [152]:
pre = """
Analyze visual representations of literature works and fictional characters. For each visual representation, identify which of these methods is used:

1. Illustration - Detailed artistic depiction of scenes, characters, or moments from literature works
2. Cartoon - Simplified or exaggerated drawing style for humorous effect
3. Caricature - Intentionally distorted features of characters for comic or satirical purposes
4. Emoji Combination - Multiple emojis merged to represent literary elements
5. Minimalism - Simplified visual representation using minimal design elements
6. Other - If none above, specify the visual method used

Rules:
- Choose only ONE method per visual representation
- If multiple methods seem applicable, select the one that appears earlier in the list
- For methods 1-5, respond with just the method name
- For method 6 (Other), provide the specific visual/artistic term that describes the method used

Example scenarios:
- A detailed painting of Alice falling down the rabbit hole → Illustration
- A whimsical, simplified drawing of Don Quixote fighting windmills → Cartoon
- An exaggerated sketch of Captain Ahab's obsessive expression → Caricature
- 🐺 + 👵 to represent Little Red Riding Hood → Emoji Combination
- Single line drawing to represent Odysseus's journey → Minimalism
- Abstract shapes representing the transformation in Kafka's Metamorphosis → Other (Abstract Symbolism)
"""